# 02 - Activation patching and the ablation harness

*Evaluating the Faithfulness of Attribution-Based Interpretability Tooling in GPT-2*

This notebook builds the two tools the rest of the study depends on.

The first is activation patching, which measures how important each attention head is by intervening on the model and watching what happens to the output. It is treated as the reference method because it changes the computation directly rather than estimating the effect.

The second is a mean-ablation harness, which switches heads off so their necessity can be tested later.

Method follows Meng et al. (2022) and Wang et al. (2023). Implemented with TransformerLens hooks (Nanda and Bloom, 2022).

In [ ]:
!pip -q install transformer_lens

In [ ]:
import torch
import random
import numpy as np
import matplotlib.pyplot as plt
from transformer_lens import HookedTransformer, utils

torch.set_grad_enabled(False)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2", device=device)

n_layers = model.cfg.n_layers
n_heads = model.cfg.n_heads
print("GPT-2 small:", n_layers, "layers x", n_heads, "heads | device:", device)

## Clean and corrupted sentence pairs

Patching needs two versions of each sentence that differ as little as possible.

In the clean version the second name acts as the subject, so the first name is the correct answer. In the corrupted version the roles are swapped, which flips the correct answer. The two versions differ by a single token, so any change in the model's behaviour can be put down to that token rather than to some other difference in wording.

In [ ]:
templates = [
    "When{A} and{B} went to the shop,{S} gave a drink to",
    "When{A} and{B} got to the office,{S} sent a letter to",
]

candidate_names = [" Mary", " John", " Tom", " James", " Anna", " Kate",
                   " Mark", " Paul", " Alice", " Sarah", " David", " Emma"]

names = []
for name in candidate_names:
    if model.to_tokens(name, prepend_bos=False).shape[1] == 1:
        names.append(name)

all_pairs = []
for name_a in names:
    for name_b in names:
        if name_a != name_b:
            all_pairs.append((name_a, name_b))

random.seed(0)
random.shuffle(all_pairs)
pairs = all_pairs[:50]

clean_prompts = []
corrupted_prompts = []
io_tokens = []
s_tokens = []

for name_a, name_b in pairs:
    for template in templates:
        clean_prompts.append(template.format(A=name_a, B=name_b, S=name_b))
        corrupted_prompts.append(template.format(A=name_a, B=name_b, S=name_a))
        io_tokens.append(model.to_single_token(name_a))
        s_tokens.append(model.to_single_token(name_b))

clean_tokens = model.to_tokens(clean_prompts)
corrupted_tokens = model.to_tokens(corrupted_prompts)
N = len(clean_prompts)

print("sentences:", N)
print("clean:    ", repr(clean_prompts[0]))
print("corrupted:", repr(corrupted_prompts[0]))

## The metric and the two baselines

The clean baseline should be clearly positive and the corrupted baseline negative. The gap between them is the scale used to normalise every importance score that follows.

In [ ]:
def mean_logit_difference(logits):
    """Average of (correct logit - incorrect logit) across the sentences."""
    final_logits = logits[:, -1, :]

    total = 0.0
    for i in range(N):
        correct_logit = final_logits[i, io_tokens[i]]
        wrong_logit = final_logits[i, s_tokens[i]]
        total = total + (correct_logit - wrong_logit).item()

    return total / N

In [ ]:
# run_with_cache keeps a copy of the internal activations so they can be patched in later
clean_logits, clean_cache = model.run_with_cache(clean_tokens)
corrupted_logits = model(corrupted_tokens)

CLEAN_BASELINE = mean_logit_difference(clean_logits)
CORRUPTED_BASELINE = mean_logit_difference(corrupted_logits)
SCALE = CLEAN_BASELINE - CORRUPTED_BASELINE

print("clean baseline:    ", round(CLEAN_BASELINE, 3))
print("corrupted baseline:", round(CORRUPTED_BASELINE, 3))
print("scale (difference):", round(SCALE, 3))

## Activation patching

For each attention head in turn:

1. run the model on the corrupted sentence
2. replace that one head's output with its output from the clean run
3. measure how far the logit difference moves back towards the clean value

The result is a restoration score. A score of 1 means that head on its own fully restores the behaviour, 0 means it does nothing, and a negative score means it pushes the model further towards the wrong answer.

The direction matters: clean activations are inserted into a corrupted run, so this measures restoration. Patching the other way round would measure disruption instead.

The hook below reads `head_to_patch`, which is set just before each run. `hook.name` tells the hook which layer it is attached to, so the matching clean activations can be looked up.

In [ ]:
head_to_patch = 0   # set before each run

def patch_one_head(z, hook):
    # z has shape [sentence, position, head, head_dimension]
    # replace one head's output with the clean version, at every position
    z[:, :, head_to_patch, :] = clean_cache[hook.name][:, :, head_to_patch, :]
    return z

In [ ]:
patching_scores = np.zeros((n_layers, n_heads))

for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    for head in range(n_heads):
        head_to_patch = head
        patched_logits = model.run_with_hooks(corrupted_tokens, fwd_hooks=[(hook_name, patch_one_head)])
        patched_value = mean_logit_difference(patched_logits)
        patching_scores[layer, head] = (patched_value - CORRUPTED_BASELINE) / SCALE

print("patched", n_layers * n_heads, "heads, one at a time")

In [ ]:
largest = abs(patching_scores).max()

plt.figure(figsize=(8, 6))
plt.imshow(patching_scores, cmap="RdBu", vmin=-largest, vmax=largest)
plt.colorbar(label="restoration score")
plt.xlabel("head")
plt.ylabel("layer")
plt.title("Activation patching importance per head")
plt.xticks(range(n_heads))
plt.yticks(range(n_layers))
plt.show()

In [ ]:
# put every head into one list so it can be sorted by score
scored_heads = []
for layer in range(n_layers):
    for head in range(n_heads):
        scored_heads.append((patching_scores[layer, head], layer, head))

scored_heads.sort(reverse=True)

print("top 10 heads by restoration score")
for score, layer, head in scored_heads[:10]:
    print("  layer", layer, "head", head, "->", round(score, 3))

A small number of heads should stand out, mostly in the middle and later layers, rather than importance being spread evenly. Those heads should overlap with the circuit published by Wang et al. (2023), which notebook 04 checks properly.

## The ablation harness

To ablate a head, its output is replaced with the average output of that head across the dataset, calculated separately at each position.

The average is used rather than zero because zeroing pushes the model into a state it never sees during training, so any drop in performance might reflect that rather than the head's actual role. The average removes the information specific to the current sentence while keeping the usual scale of the activations.

This is a fairly gentle intervention, so behaviour is expected to degrade gradually rather than disappear.

In [ ]:
# average activation for each layer, keeping the position dimension
mean_activations = {}
for layer in range(n_layers):
    hook_name = utils.get_act_name("z", layer)
    mean_activations[layer] = clean_cache[hook_name].mean(0, keepdim=True)

# lets a hook work out which layer it is attached to
layer_of_hook = {}
for layer in range(n_layers):
    layer_of_hook[utils.get_act_name("z", layer)] = layer

heads_to_ablate = []   # list of (layer, head), set before each run

def ablate_heads(z, hook):
    layer = layer_of_hook[hook.name]
    for target_layer, target_head in heads_to_ablate:
        if target_layer == layer:
            z[:, :, target_head, :] = mean_activations[layer][:, :, target_head, :]
    return z

def run_with_ablation(heads):
    """Mean-ablate the given (layer, head) pairs and return the mean logit difference."""
    global heads_to_ablate
    heads_to_ablate = heads

    hooks = []
    for layer in range(n_layers):
        hooks.append((utils.get_act_name("z", layer), ablate_heads))

    return mean_logit_difference(model.run_with_hooks(clean_tokens, fwd_hooks=hooks))

### Checking the harness works

Ablating the heads that patching says matter should hurt performance more than ablating the same number of randomly chosen heads.

In [ ]:
top_heads = []
for score, layer, head in scored_heads[:8]:
    top_heads.append((layer, head))

all_heads = []
for layer in range(n_layers):
    for head in range(n_heads):
        all_heads.append((layer, head))

random.seed(1)
random_heads = random.sample(all_heads, 8)

print("nothing ablated:        ", round(CLEAN_BASELINE, 3))
print("top 8 patching heads:   ", round(run_with_ablation(top_heads), 3))
print("8 randomly chosen heads:", round(run_with_ablation(random_heads), 3))

## Result

Two things are now in place: a restoration score for every attention head, produced by direct intervention, and a harness that can switch any set of heads off and measure what happens.

Ablating the top-ranked heads costs more than ablating random ones, which is what would be expected if patching is finding genuinely important components. The drop is partial rather than total, which fits a circuit whose work is spread across several heads and a mean ablation that is deliberately gentle.

Notebook 03 produces the same kind of score using the faster gradient-based method, and notebook 04 compares the two.

### References

- Meng, K. et al. (2022) *Locating and Editing Factual Associations in GPT*. NeurIPS.
- Nanda, N. and Bloom, J. (2022) *TransformerLens*.
- Wang, K. et al. (2023) *Interpretability in the Wild: a Circuit for Indirect Object Identification in GPT-2 small*. ICLR.